In [10]:
## imports for XGBoost multi-class classification
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import os

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)

np.random.seed(42)

## loading preprocessed data (from notebook 02) + reconstruction errors (from notebook 03)
X_train = np.load('../data/processed/X_train.npy')
X_test  = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test  = np.load('../data/processed/y_test.npy')

recon_error_train = np.load('../data/processed/recon_error_train.npy')
recon_error_test  = np.load('../data/processed/recon_error_test.npy')

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"recon_error_train: {recon_error_train.shape}, recon_error_test: {recon_error_test.shape}")

label_map = {0: 'Normal', 1: 'DoS', 2: 'Probe', 3: 'R2L', 4: 'U2R'}
print("\nClass distribution (train):")
for cid, cname in label_map.items():
    print(f"  {cname:8s}: {(y_train == cid).sum()}")

X_train: (125973, 122), X_test: (22251, 122)
recon_error_train: (125973,), recon_error_test: (22251,)

Class distribution (train):
  Normal  : 67343
  DoS     : 45927
  Probe   : 11656
  R2L     : 995
  U2R     : 52


In [11]:
## building both feature sets for the ablation study
# Version A — baseline: 122 features only, no autoencoder signal
X_train_A = X_train
X_test_A  = X_test

# Version B — hybrid: 123 features, recon error injected as the last column
# reshape recon_error from (n,) to (n,1) so it can be hstacked as a column
X_train_B = np.hstack([X_train, recon_error_train.reshape(-1, 1)])
X_test_B  = np.hstack([X_test, recon_error_test.reshape(-1, 1)])

print(f"Version A (baseline) — X_train_A: {X_train_A.shape}, X_test_A: {X_test_A.shape}")
print(f"Version B (hybrid)   — X_train_B: {X_train_B.shape}, X_test_B: {X_test_B.shape}")

# sanity check: last column of Version B should exactly equal recon_error_train
assert np.allclose(X_train_B[:, -1], recon_error_train), "Injection mismatch!"
print("\nSanity check passed: last column of X_train_B matches recon_error_train")

Version A (baseline) — X_train_A: (125973, 122), X_test_A: (22251, 122)
Version B (hybrid)   — X_train_B: (125973, 123), X_test_B: (22251, 123)

Sanity check passed: last column of X_train_B matches recon_error_train


In [12]:
## computing sample weights to counter severe class imbalance
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print("Sample weight per class (should be higher for rarer classes):")
for cid, cname in label_map.items():
    w = sample_weights[y_train == cid][0]  # weight is identical within a class
    print(f"  {cname:8s} (n={int((y_train==cid).sum()):6d}): weight = {w:.4f}")

print(f"\nsample_weights shape: {sample_weights.shape}")
print(f"Ratio of U2R weight to Normal weight: {sample_weights[y_train==4][0] / sample_weights[y_train==0][0]:.1f}x")

Sample weight per class (should be higher for rarer classes):
  Normal   (n= 67343): weight = 0.3741
  DoS      (n= 45927): weight = 0.5486
  Probe    (n= 11656): weight = 2.1615
  R2L      (n=   995): weight = 25.3212
  U2R      (n=    52): weight = 484.5115

sample_weights shape: (125973,)
Ratio of U2R weight to Normal weight: 1295.1x


In [13]:
## Version A — baseline XGBoost, 122 features (no reconstruction error)
xgb_A = XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    eval_metric='mlogloss',
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_A.fit(X_train_A, y_train, sample_weight=sample_weights)

## evaluate on test set
y_pred_A = xgb_A.predict(X_test_A)
f1_macro_A = f1_score(y_test, y_pred_A, average='macro')

print(f"Version A (baseline, 122 features) — Macro F1: {f1_macro_A:.4f}\n")
print(classification_report(y_test, y_pred_A, target_names=list(label_map.values())))

Version A (baseline, 122 features) — Macro F1: 0.6183

              precision    recall  f1-score   support

      Normal       0.71      0.97      0.82      9711
         DoS       0.97      0.87      0.92      7167
       Probe       0.82      0.74      0.78      2421
         R2L       0.99      0.13      0.23      2885
         U2R       0.64      0.24      0.35        67

    accuracy                           0.80     22251
   macro avg       0.83      0.59      0.62     22251
weighted avg       0.84      0.80      0.77     22251



In [14]:
## Version B — hybrid XGBoost, 123 features (with reconstruction error injected)
xgb_B = XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    eval_metric='mlogloss',
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_B.fit(X_train_B, y_train, sample_weight=sample_weights)

## evaluate on test set
y_pred_B = xgb_B.predict(X_test_B)
f1_macro_B = f1_score(y_test, y_pred_B, average='macro')

print(f"Version B (hybrid, 123 features) — Macro F1: {f1_macro_B:.4f}\n")
print(classification_report(y_test, y_pred_B, target_names=list(label_map.values())))

## direct comparison
print(f"\n{'='*50}")
print(f"ABLATION RESULT")
print(f"{'='*50}")
print(f"Version A (baseline, 122 features): Macro F1 = {f1_macro_A:.4f}")
print(f"Version B (hybrid,   123 features): Macro F1 = {f1_macro_B:.4f}")
print(f"Improvement: {(f1_macro_B - f1_macro_A):+.4f} ({(f1_macro_B - f1_macro_A)/f1_macro_A*100:+.2f}%)")

Version B (hybrid, 123 features) — Macro F1: 0.6382

              precision    recall  f1-score   support

      Normal       0.74      0.97      0.84      9711
         DoS       0.97      0.88      0.92      7167
       Probe       0.83      0.85      0.84      2421
         R2L       0.98      0.15      0.26      2885
         U2R       0.62      0.22      0.33        67

    accuracy                           0.82     22251
   macro avg       0.83      0.61      0.64     22251
weighted avg       0.85      0.82      0.79     22251


ABLATION RESULT
Version A (baseline, 122 features): Macro F1 = 0.6183
Version B (hybrid,   123 features): Macro F1 = 0.6382
Improvement: +0.0199 (+3.22%)


In [15]:
## 5-fold stratified CV comparing Version A vs Version B
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores_A = []
cv_scores_B = []

xgb_params = dict(
    objective='multi:softmax',
    num_class=5,
    eval_metric='mlogloss',
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_A, y_train), 1):
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    # recompute sample weights fresh for this fold's training subset
    fold_weights = compute_sample_weight(class_weight='balanced', y=y_tr)

    # --- Version A (122 features) ---
    model_A = XGBClassifier(**xgb_params)
    model_A.fit(X_train_A[train_idx], y_tr, sample_weight=fold_weights)
    pred_A = model_A.predict(X_train_A[val_idx])
    score_A = f1_score(y_val, pred_A, average='macro')
    cv_scores_A.append(score_A)

    # --- Version B (123 features) ---
    model_B = XGBClassifier(**xgb_params)
    model_B.fit(X_train_B[train_idx], y_tr, sample_weight=fold_weights)
    pred_B = model_B.predict(X_train_B[val_idx])
    score_B = f1_score(y_val, pred_B, average='macro')
    cv_scores_B.append(score_B)

    print(f"Fold {fold}: A={score_A:.4f}  B={score_B:.4f}  diff={score_B-score_A:+.4f}")

cv_scores_A = np.array(cv_scores_A)
cv_scores_B = np.array(cv_scores_B)

print(f"\n{'='*50}")
print(f"Version A — mean: {cv_scores_A.mean():.4f}, std: {cv_scores_A.std():.4f}")
print(f"Version B — mean: {cv_scores_B.mean():.4f}, std: {cv_scores_B.std():.4f}")
print(f"Mean difference (B - A): {(cv_scores_B - cv_scores_A).mean():+.4f}")
print(f"Version B beat Version A in {(cv_scores_B > cv_scores_A).sum()}/5 folds")

Fold 1: A=0.9379  B=0.9391  diff=+0.0011
Fold 2: A=0.9436  B=0.9558  diff=+0.0122
Fold 3: A=0.9888  B=0.9806  diff=-0.0082
Fold 4: A=0.9685  B=0.9534  diff=-0.0152
Fold 5: A=0.9423  B=0.9463  diff=+0.0041

Version A — mean: 0.9562, std: 0.0195
Version B — mean: 0.9550, std: 0.0140
Mean difference (B - A): -0.0012
Version B beat Version A in 3/5 folds


In [16]:
## GridSearchCV hyperparameter tuning — Version A (122 features, baseline)
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6],
    'learning_rate': [0.1, 0.2],
}

base_xgb_A = XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    eval_metric='mlogloss',
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=1   # single-threaded per model — GridSearchCV parallelizes across combos instead
)

grid_A = GridSearchCV(
    estimator=base_xgb_A,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_A.fit(X_train_A, y_train, sample_weight=sample_weights)

print(f"\nBest params (Version A): {grid_A.best_params_}")
print(f"Best CV macro F1 (Version A): {grid_A.best_score_:.4f}")

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=100; total time=  13.0s
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=100; total time=  13.1s
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=100; total time=  13.2s
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=100; total time=  15.3s
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=100; total time=  15.5s
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=200; total time=  25.0s
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=200; total time=  25.0s
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=200; total time=  25.3s
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=100; total time=  15.0s
[CV] END ...learning_rate=0.2, max_depth=4, n_estimators=100; total time=  12.6s
[CV] END ...learning_rate=0.2, max_depth=4, n_estimators=100; total time=  13.0s
[CV] END ...learning_rate=0.2, max_depth=4, n_est

In [17]:
## GridSearchCV hyperparameter tuning — Version B (123 features, hybrid)
base_xgb_B = XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    eval_metric='mlogloss',
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=1
)

grid_B = GridSearchCV(
    estimator=base_xgb_B,
    param_grid=param_grid,   # same grid as Cell 7
    scoring='f1_macro',
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_B.fit(X_train_B, y_train, sample_weight=sample_weights)

print(f"\nBest params (Version B): {grid_B.best_params_}")
print(f"Best CV macro F1 (Version B): {grid_B.best_score_:.4f}")

## direct comparison of tuned models
print(f"\n{'='*50}")
print(f"TUNED COMPARISON (3-fold CV, within train distribution)")
print(f"{'='*50}")
print(f"Version A best: {grid_A.best_score_:.4f}  (params: {grid_A.best_params_})")
print(f"Version B best: {grid_B.best_score_:.4f}  (params: {grid_B.best_params_})")
print(f"Difference (B - A): {(grid_B.best_score_ - grid_A.best_score_):+.4f}")

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=100; total time=  15.0s
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=100; total time=  15.1s
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=100; total time=  15.1s
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=100; total time=  17.7s
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=100; total time=  18.2s
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=200; total time=  30.7s
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=200; total time=  30.9s
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=200; total time=  30.9s
[CV] END ...learning_rate=0.2, max_depth=4, n_estimators=100; total time=  15.5s
[CV] END ...learning_rate=0.1, max_depth=6, n_estimators=100; total time=  19.0s
[CV] END ...learning_rate=0.2, max_depth=4, n_estimators=100; total time=  15.2s
[CV] END ...learning_rate=0.2, max_depth=4, n_est